# Make Skims

Create skims between 2020 Census tracts within Chicago.  Start from GTFS files for April 2025, corresponding to survey data. 

Computes transit travel times between every pair of in-scope tracts, for five weekday time
periods, using [r5py](https://r5py.readthedocs.io/) over the CTA, Pace and Metra GTFS feeds.

**Prerequisites**
- The `rh_pricing` conda environment (see `README.md`) — r5py needs Java 21.
- `data/cook_county.osm.pbf` — run `Download OSM.ipynb` first to create it.
- The three GTFS feeds and `tract_2020_district.csv` in `data/`.

Outputs one Parquet file per time period in `out/`, each a long table of
`from_id, to_id, travel_time_min`.

In [ ]:
import os, datetime
import geopandas as gpd
import pandas as pd
from r5py import TransportNetwork, TravelTimeMatrixComputer, TransportMode

DATA_DIR = "data"
OUT_DIR  = "out"

OSM_PBF    = f"{DATA_DIR}/cook_county.osm.pbf"
GTFS_FILES = [
    f"{DATA_DIR}/gtfs-cta-202504170110.zip",
    f"{DATA_DIR}/gtfs-metra-202510301823.zip",
    f"{DATA_DIR}/gtfs-pace-202502270021.zip",
]
TRACTS_CSV = f"{DATA_DIR}/tract_2020_district.csv"

# Service date: a representative weekday. April 16, 2025 is a Wednesday, and all three
# feeds have regular weekday service in effect on that date.
SERVICE_DATE = datetime.date(2025, 4, 16)

# Five time periods as (name, start time, duration). r5py samples departures across the
# whole window and reports the median travel time, so each skim reflects average service
# over that period. 'night' spans 10pm-6am and crosses midnight into the next (still
# weekday) morning.
TIME_PERIODS = [
    ("night",   datetime.time(22, 0), datetime.timedelta(hours=8)),
    ("am_peak", datetime.time( 6, 0), datetime.timedelta(hours=3)),
    ("midday",  datetime.time( 9, 0), datetime.timedelta(hours=7)),
    ("pm_peak", datetime.time(16, 0), datetime.timedelta(hours=3)),
    ("evening", datetime.time(19, 0), datetime.timedelta(hours=3)),
]

# Trips longer than this are reported as unreachable (NaN).
MAX_TIME = datetime.timedelta(hours=3)

# For a quick test run, set to an int (e.g. 25) to limit the number of origins. None = all.
MAX_ORIGINS = None

os.makedirs(OUT_DIR, exist_ok=True)

## 1. Load tract centroids (origins & destinations)

Read the tract equivalency file, drop `External` tracts (outside the City of Chicago), and
build point origins/destinations from the centroid lat-lon in the CRS r5py expects (EPSG:4326).

In [ ]:
tracts = pd.read_csv(TRACTS_CSV, dtype={"GEOID": str})
tracts = tracts[tracts["RH_Zone"] != "External"].copy()

tracts["lat"] = tracts["INTPTLAT"].astype(float)
tracts["lon"] = tracts["INTPTLON"].astype(float)

points = gpd.GeoDataFrame(
    {"id": tracts["GEOID"].values},
    geometry=gpd.points_from_xy(tracts["lon"], tracts["lat"]),
    crs="EPSG:4326",
)

origins = points if MAX_ORIGINS is None else points.iloc[:MAX_ORIGINS].copy()
print(f"{len(points)} in-scope Chicago tracts; using {len(origins)} origins x {len(points)} destinations")
origins.head()

## 2. Build the multimodal transport network

This loads the OSM street network and all three GTFS feeds into a single routable network.
The first build is the slow step (a few minutes) and is memory-hungry; r5py/R5 claims up to
80% of available RAM by default.

In [ ]:
transport_network = TransportNetwork(OSM_PBF, GTFS_FILES)

## 3. Compute a travel-time skim for each time period

For each period, r5py runs RAPTOR for departures sampled across the whole window and returns
the **median** travel time per OD pair (in minutes; `NaN` if unreachable within `MAX_TIME`).
Wider windows (night, midday) sample more departure minutes and take proportionally longer —
start with `MAX_ORIGINS` set to a small number to smoke-test before the full run.

In [ ]:
def compute_period(name, start_time, window):
    departure = datetime.datetime.combine(SERVICE_DATE, start_time)
    print(f"[{name}] departure {departure}, window {window} ...")

    computer = TravelTimeMatrixComputer(
        transport_network,
        origins=origins,
        destinations=points,
        departure=departure,
        departure_time_window=window,
        max_time=MAX_TIME,
        transport_modes=[TransportMode.TRANSIT, TransportMode.WALK],
        snap_to_network=True,
    )
    skim = computer.compute_travel_times().rename(columns={"travel_time": "travel_time_min"})

    out_path = f"{OUT_DIR}/transit_skim_{name}.parquet"
    skim.to_parquet(out_path, index=False)

    reachable = skim["travel_time_min"].notna().mean() * 100
    print(f"[{name}] {len(skim):,} OD pairs, {reachable:.1f}% reachable -> {out_path}")
    return skim

skims = {name: compute_period(name, start, window) for name, start, window in TIME_PERIODS}

## 4. Quick look

In [ ]:
am = skims["am_peak"]
print("AM peak median travel time (min):", round(am["travel_time_min"].median(), 1))
am.head()